# New RRL Results

In this notebook I will attempt to automate the pulled out RRLs from new_RRL_search to see if we can confirm their classification

## Importing Libraries and Data

In [1]:
#import libraries and set up tap service
import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table
import pyvo
import astropy.units as u
from lsst.rsp import get_tap_service
import pandas as pd
from astropy.timeseries import LombScargle, LombScargleMultiband
pd.set_option('display.max_columns', None)

from lsst.utils.plotting import (get_multiband_plot_colors,
                                 get_multiband_plot_symbols)

from lsst.daf.butler import Butler
from lsst.utils.plotting import (
    get_multiband_plot_colors,
    get_multiband_plot_symbols,
    get_multiband_plot_linestyles
)

service = get_tap_service("tap")
assert service is not None

import seaborn as sns
my_pal = sns.color_palette("colorblind")

import warnings
# This silences the ResourceWarning about unclosed sockets
warnings.filterwarnings("ignore", category=ResourceWarning)

import lsst.afw.display as afwDisplay
import lsst.geom as geom
from astropy.wcs import WCS
import numpy as np
import gc

In [2]:
Ex_diaObj_df = pd.read_csv("data/Extra_diaObj_df.csv",index_col=0)
# forcedDiaObj_df = pd.read_csv("data/diaForced_df.csv",index_col=0)

# grouped_forcedDia = forcedDiaObj_df.groupby("diaObjectId")

search_df = pd.read_csv("data/rrl_search_df.csv",index_col=0)
forcedSearch_df = pd.read_csv("data/rrl_forcedSource_df.csv",index_col=0)

grouped_forcedSearch = forcedSearch_df.groupby("ObjectId")

diaSearch_df = pd.read_csv("data/rrl_diaSource_df.csv",index_col=0)

grouped_diaSearch = diaSearch_df.groupby("diaObjectId")


filter_names = ['u', 'g', 'r', 'i', 'z', 'y']
filter_colors = get_multiband_plot_colors()
filter_symbols = get_multiband_plot_symbols()
filter_linestyles = get_multiband_plot_linestyles()

In [3]:
#adding in center point of fields from 301_0_DP1
field_centers = {
    "47 Tuc": [6.02, -72.08],
    "SV 38 7": [37.86, 6.98],
    "Fornax": [40.00, -34.45],
    "ECDFS": [53.13, -28.10],
    "EDFS": [59.10, -48.73],
    "SV 95 -25": [95.00, -25.00],
    "Seagull": [106.23, -10.51],
}
field_names = list(field_centers.keys())

In [4]:
afwDisplay.setDefaultBackend("matplotlib")

butler = Butler("dp1", collections="LSSTComCam/DP1")
assert butler is not None

In [5]:
#load in the maglim for each object
u_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='u')
g_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='g')
r_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='r')
i_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='i')
z_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band='z')
y_maglim = butler.get('deepCoadd_psf_maglim_consolidated_map_weighted_mean',
                      band="y")

## Defining Loc Functions

In [6]:
def remove_figure(fig):
    """
    Remove a figure to reduce memory footprint.

    Parameters
    ----------
    fig: matplotlib.figure.Figure
        Figure to be removed.

    Returns
    -------
    None
    """

    for ax in fig.get_axes():
        for im in ax.get_images():
            im.remove()

    fig.clf()
    plt.close(fig)
    gc.collect()

In [7]:
def zoom_in(ax, wcs, ra, dec, side):
    """
    Zoom in on the search location.

    Parameters
    ----------
    ax: matplotlib.axes._axes.Axes
        Axes for showing a plot.
    wcs: lsst.afw.geom.SkyWcs
        WCS of an image.
    ra: float
        RA of a celestial location in degree.
    dec: float
        DEC of a celestial location in degree.
    side: float
        Size of the display region in arcseconds.

    Returns
    -------
    None
    """

    x, y = wcs.skyToPixel(geom.SpherePoint(ra, dec, geom.degrees))
    pixel_scale = 0.2
    side_pix = side/pixel_scale
    ax.set_xlim([x - side_pix/2, x + side_pix/2])
    ax.set_ylim([y - side_pix/2, y + side_pix/2])

In [8]:
def make_plot(deep_coadd,row,tx,tx2,objects,objects2,directory=""):

    diaObjectId = row["diaObjectId"]
    ra_val = row["ra"]
    dec_val = row["dec"]

    deep_coadd_extent = (deep_coadd.getBBox().beginX,
                     deep_coadd.getBBox().endX,
                     deep_coadd.getBBox().beginY,
                     deep_coadd.getBBox().endY)

    fig = plt.figure(figsize=(10, 10))
    ax1 = fig.add_subplot(121, projection=WCS(deep_coadd.getWcs().getFitsMetadata()))
    
    
    im = ax1.imshow(deep_coadd.image.array, cmap='gray',
                    vmin=-75, vmax=125,
                    extent=deep_coadd_extent,
                    origin='lower')
    
    ax1.scatter(objects[tx]['r_centroid_x'], objects[tx]['r_centroid_y'],
                facecolors='none', edgecolors='orange',label="Low SNR Obj")

    ax1.scatter(objects2[tx2]['r_centroid_x'], objects2[tx2]['r_centroid_y'],
                facecolors='none', edgecolors='red',label="High SNR Obj")
    
    
    # Calculate the center pixel
    center_x = (deep_coadd_extent[1] + deep_coadd_extent[0]) / 2
    center_y = (deep_coadd_extent[-1] + deep_coadd_extent[-2]) / 2
    
    # Plot it (using the same coordinate system as your objects)
    # ax1.scatter(center_x, center_y, color='cyan', marker='x', s=100, label='Pixel Center')

    ax1.scatter(ra_val, dec_val, 
            transform=ax1.get_transform('world'), 
            marker='+', 
            s=300, 
            color='lime', 
            label=f'diaObj RA/Dec:\n{ra_val:.2f}/{dec_val:.2f}')
    
    # Access the RA and Dec coordinate helpers from the axes
    ra_scale = ax1.coords[0]
    dec_scale = ax1.coords[1]

    
    # Set the format to decimal degrees ('.degree' or 'd.dd')
    ra_scale.set_major_formatter('d.dd')
    dec_scale.set_major_formatter('d.dd')
    
    # Optional: Adjust the number of ticks if it gets crowded
    ra_scale.set_ticks(number=5)
    dec_scale.set_ticks(number=5)
    
    ax1.set_xlabel('Right Ascension',fontsize=14)
    ax1.set_ylabel('Declination',fontsize=14)
    
    for ax in [ax1]:
        ax.grid(color='white', ls='solid')

    side = 140

    zoom_in(ax1, deep_coadd.getWcs(), ra_val, dec_val, side)

    plt.title(f"diaObjectId: {diaObjectId}\n(In_Obj: {row["in_Obj"]})",fontsize=16)
    plt.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
    plt.tight_layout()
    if directory:
        plt.savefig(f"{directory}/coadd_image.png",dpi=330,bbox_inches="tight")
    else:
        plt.show()
        pass
    remove_figure(fig)

In [9]:
def make_field_plot(row,path=""):
    name = row["DR3_field_loc_center"]
    ra_cen, dec_cen = field_centers[name]
    span_dec = 0.75
    if name == 'SV 38 7':
        span_dec = 1.00
    span_ra = span_dec / np.cos(np.deg2rad(dec_cen))
    ra = np.linspace(ra_cen-span_ra, ra_cen+span_ra, 250)
    dec = np.linspace(dec_cen-span_dec, dec_cen+span_dec, 250)

    x, y = np.meshgrid(ra, dec)
    values = r_maglim.get_values_pos(x, y)
    row1, col = np.where(values < 0.0)
    values[row1, col] = 'NaN'
    plt.pcolormesh(x, y, values, vmin=24.0, vmax=27)
    plt.colorbar(label="PSF Mag Limit (r-band)")

    size = 20

    conversion_mas_deg = 1 / 3600000

    current_stars = Ex_diaObj_df[Ex_diaObj_df["DR3_field_loc_center"] == name]
    plt.errorbar(current_stars["DR3_ra"],current_stars["DR3_dec"],current_stars["DR3_ra_error"] * conversion_mas_deg,
                 current_stars["DR3_dec_error"] * conversion_mas_deg,fmt="none",elinewidth=size/2,c="k")
    plt.scatter(current_stars["DR3_ra"],current_stars["DR3_dec"],s=size,c="k",label=name)
    plt.scatter(row["ra"],row["dec"],s=size*2,c="y",marker="*",label="Current Star")

    plt.xlabel("Right Ascension (deg)",fontsize=16)
    plt.ylabel("Declination (deg)",fontsize=16)
    plt.title(f"diaObjectId: {row["diaObjectId"]}",fontsize=20)
    plt.legend()
    plt.gca().invert_xaxis()
    plt.tight_layout()
    if path:
        plt.savefig(f"{path}/{name}_field_plot.png",dpi=330,bbox_inches="tight")
    else:
        plt.show()
    plt.close()

In [10]:
def pull_out_row(row,path=""):
    print(f"Making location plots for {int(row["diaObjectId"])}...")

    """
    Making Coadd Plot for Star
    """
    
    ra,dec = row[["ra","dec"]]

    band = "r"

    query = "band.name = :band AND patch.region OVERLAPS POINT(:ra, :dec)"
    bind = {'band': band, 'ra': ra, 'dec': dec}
    deep_coadd_refs = butler.query_datasets("deep_coadd", where=query,
                                            bind=bind, order_by='patch')

    for ref in deep_coadd_refs:
        dataId = ref.dataId
        use_columns = ['objectId', 'patch',
                   'r_centroid_x', 'r_centroid_y',
                   'r_psfFlux', 'r_psfFluxErr',"coord_ra","coord_dec"]
        objects = butler.get('object', tract=dataId.get('tract'),
                             parameters={'columns': use_columns})

        filt_objects = objects[objects['patch'] == dataId.get('patch')]
        
        if np.min(filt_objects["coord_ra"]) < ra and ra < np.max(filt_objects["coord_ra"]):
            if np.min(filt_objects["coord_dec"]) < dec and dec < np.max(filt_objects["coord_dec"]):
                deep_coadd = butler.get(ref)
                break


    deep_coadd_extent = (deep_coadd.getBBox().beginX,
                     deep_coadd.getBBox().endX,
                     deep_coadd.getBBox().beginY,
                     deep_coadd.getBBox().endY)



    use_columns = ['objectId', 'patch',
               'r_centroid_x', 'r_centroid_y',
               'r_psfFlux', 'r_psfFluxErr']
    
    objects = butler.get('object', tract=dataId.get('tract'),
                     parameters={'columns': use_columns})

    tx = np.where(objects['patch'] == dataId.get('patch'))[0]
    # print("Number of objects: ", len(tx))

    sel = objects['r_psfFlux'] / objects['r_psfFluxErr'] > 300
    objects2 = objects[sel]
    tx2 = np.where(objects2['patch'] == dataId.get('patch'))[0]
    # print("Number of high SNR objects: ", len(tx2))
    # print(f"Stellar ra/dec: {ra} {dec}")
    # print(f"in_Obj: {row["in_Obj"]}")

    if path:
        # Creating New Folder for current star
        dia_path = os.path.join(path,f"{row["diaObjectId"]}")
        
        if not os.path.isdir(dia_path):
            os.mkdir(dia_path)
    
    make_plot(deep_coadd,row,tx,tx2,objects,objects2,directory=dia_path)

    """
    Making Distribution Plots with Star Highlighted
    """
    
    make_field_plot(row,dia_path)


## Defining Lomb Functions

In [11]:
def first_fit(row,group_data,path="",filt=100):
    results = group_data.reset_index()

    diaId = row["diaObjectId"]

    minfreq = 1 / (1.0*u.d)
    maxfreq = 1 / (0.05*u.d)
    
    obj_mjd_days = np.array(results['expMidptMJD']) * u.day
    obj_fluxes = np.array(results['psfFlux'])
    obj_flux_errs = np.array(results['psfFluxErr'])
    obj_frequency, obj_power =\
        LombScargleMultiband(obj_mjd_days, obj_fluxes,
                             results['band'], obj_flux_errs).autopower(minimum_frequency=minfreq,
                                                                       maximum_frequency=maxfreq,
                                                                       samples_per_peak=20)
    max_power = np.argmax(obj_power)
    obj_freq = obj_frequency[max_power]
    obj_period = 1.0 / obj_freq
    # print(f"period: {obj_period: .8F}")

    t0 = 0.0
    obj_mjd_norm = (obj_mjd_days.value - t0) / obj_period.value
    obj_phase = np.mod(obj_mjd_norm, 1.0)

    phase = pd.Series(obj_phase)
    
    mag = -2.5 * np.log10(pd.Series(obj_fluxes) / (3631 * 1e9))


    """
    Saving Combined
    """

    fig, ax = plt.subplots(7, 1, figsize=(7, 21))

    
    max_mag = float("-inf")
    min_mag = float("inf")
    
    ax1 = ax[0]
    for band in filter_names:
        pickband = results['band'] == band
        temp_mag = mag[pickband]

        if len(temp_mag):
            med = np.median(temp_mag)
            std3 = np.std(temp_mag) * filt
        else:
            med = 0
            std3 = 0
    
        mask = (temp_mag > med - std3) & (temp_mag < med + std3)
    
        masked_mag = temp_mag[mask]
    
        FluxErr = results[pickband]["psfFluxErr"][mask]
        FluxBand = results[pickband]["psfFlux"][mask]
    
        yerr = (2.5 / np.log(10)) * (FluxErr / FluxBand)
    
        full_phase = pd.concat([phase[pickband][mask],phase[pickband][mask] + 1])
        full_mag = pd.concat([masked_mag,masked_mag])
        full_mag_err = pd.concat([yerr,yerr])
        
        ax1.errorbar(full_phase, full_mag, yerr=full_mag_err,
                marker=filter_symbols[band], color=filter_colors[band],
                label=f"{band}", markersize=7, linestyle='none', fillstyle='none')
        
        if np.max(masked_mag) > max_mag:
            max_mag = np.max(masked_mag)
    
        if np.min(masked_mag) < min_mag:
            min_mag = np.min(masked_mag) 
        
    
    ax1.set_ylabel('PSF magnitude',fontsize=14)
    ax1.set_xlabel('Phase',fontsize=14)
    ax1.set_title(f"Overall Mag Phase Curve (Filtered)",fontsize=16)
    ax1.minorticks_on()
    ax1.legend(ncols=2, loc='lower left')

    try:
        ax1.set_ylim(max_mag + .2, min_mag - .2)
    except:
        pass
        
    ax1.set_xlim(-0.05,2.05)

    
    for i in range(1,7):
        ax1 = ax[i]
    
        band = filter_names[i-1]
    
        pickband = results['band'] == band
        temp_mag = mag[pickband]
    
        if len(temp_mag):
            med = np.median(temp_mag)
            std3 = np.std(temp_mag) * filt
        else:
            med = 0
            std3 = 0
    
        mask = (temp_mag > med - std3) & (temp_mag < med + std3)
    
        masked_mag = temp_mag[mask]
    
        FluxErr = results[pickband]["psfFluxErr"][mask]
        FluxBand = results[pickband]["psfFlux"][mask]
    
        yerr = (2.5 / np.log(10)) * (FluxErr / FluxBand)
    
        full_phase = pd.concat([phase[pickband][mask],phase[pickband][mask] + 1])
        full_mag = pd.concat([masked_mag,masked_mag])
        full_mag_err = pd.concat([yerr,yerr])
        
        ax1.errorbar(full_phase, full_mag, yerr=full_mag_err,
                marker=filter_symbols[band], color=filter_colors[band],
                label=f"{band}", markersize=7, linestyle='none', fillstyle='none')

        if len(masked_mag):
            ax1.set_ylim(np.max(masked_mag) + .2, np.min(masked_mag) - .2)
        
        ax1.set_xlim(-0.05,2.05)
        
        ax1.set_ylabel('PSF magnitude',fontsize=14)
        ax1.set_xlabel('Phase',fontsize=14)
        ax1.set_title(f"{band}-band Mag Phase Curve (Filtered)",fontsize=16)
        ax1.minorticks_on()
        ax1.legend(ncols=2, loc='lower left')
        
    plt.tight_layout()
    plt.suptitle(f"DiaObjectId - {diaId}",fontsize=20,y=1.01)

    if path:
        plt.savefig(f"{path}/combined_first_lomb.png",dpi=330,bbox_inches="tight")
        pass
    else:
        plt.show()

    plt.close()
    
    """
    Saving Overall Version
    """
    
    fig, ax1 = plt.subplots(1, 1, figsize=(7, 4))

    max_mag = float("-inf")
    min_mag = float("inf")
    
    for band in filter_names:
        pickband = results['band'] == band
        temp_mag = mag[pickband]

        if len(temp_mag):
            med = np.median(temp_mag)
            std3 = np.std(temp_mag) * filt
        else:
            med = 0
            std3 = 0
    
        mask = (temp_mag > med - std3) & (temp_mag < med + std3)
    
        masked_mag = temp_mag[mask]

        FluxErr = results[pickband]["psfFluxErr"][mask]
        FluxBand = results[pickband]["psfFlux"][mask]

        yerr = (2.5 / np.log(10)) * (FluxErr / FluxBand)

        full_phase = pd.concat([phase[pickband][mask],phase[pickband][mask] + 1])
        full_mag = pd.concat([masked_mag,masked_mag])
        full_mag_err = pd.concat([yerr,yerr])

        ax1.errorbar(full_phase, full_mag, yerr=full_mag_err,
                marker=filter_symbols[band], color=filter_colors[band],
                label=f"{band}", markersize=7, linestyle='none', fillstyle='none')
        
        if np.max(masked_mag) > max_mag:
            max_mag = np.max(masked_mag)
    
        if np.min(masked_mag) < min_mag:
            min_mag = np.min(masked_mag)

    
    ax1.set_ylabel('PSF magnitude',fontsize=14)
    ax1.set_xlabel('Phase',fontsize=14)
    ax1.set_title(f"Overall Mag Phase Curve (Filtered)",fontsize=16)
    ax1.minorticks_on()
    ax1.legend(ncols=2, loc='lower left')  


    try:
        ax1.set_ylim(max_mag + .2, min_mag - .2)
    except:
        pass
        
    ax1.set_xlim(-0.05,2.05)

    plt.suptitle(f"DiaObjectId - {diaId}",fontsize=20)
    plt.tight_layout()

    if path:
        plt.savefig(f"{path}/overall_first_lomb.png",dpi=330,bbox_inches="tight")
        pass
    else:
        plt.show()

    plt.close()

    """
    Saving Individual Bands
    """
    
    for band in filter_names:
        fig, ax1 = plt.subplots(1, 1, figsize=(7, 4))

        max_mag = float("-inf")
        min_mag = float("inf")
        
        pickband = results['band'] == band
        temp_mag = mag[pickband]

        if len(temp_mag):
            med = np.median(temp_mag)
            std3 = np.std(temp_mag) * filt
        else:
            med = 0
            std3 = 0
    
        mask = (temp_mag > med - std3) & (temp_mag < med + std3)
    
        masked_mag = temp_mag[mask]

        FluxErr = results[pickband]["psfFluxErr"][mask]
        FluxBand = results[pickband]["psfFlux"][mask]

        yerr = (2.5 / np.log(10)) * (FluxErr / FluxBand)

        full_phase = pd.concat([phase[pickband][mask],phase[pickband][mask] + 1])
        full_mag = pd.concat([masked_mag,masked_mag])
        full_mag_err = pd.concat([yerr,yerr])

        ax1.errorbar(full_phase, full_mag, yerr=full_mag_err,
                marker=filter_symbols[band], color=filter_colors[band],
                label=f"{band}", markersize=7, linestyle='none', fillstyle='none')
        
        if np.max(masked_mag) > max_mag:
            max_mag = np.max(masked_mag)
    
        if np.min(masked_mag) < min_mag:
            min_mag = np.min(masked_mag)

    
        ax1.set_ylabel('PSF magnitude',fontsize=14)
        ax1.set_xlabel('Phase',fontsize=14)
        ax1.set_title(f"Overall Mag Phase Curve (Filtered)",fontsize=16)
        ax1.minorticks_on()
        ax1.legend(ncols=2, loc='lower left')  
    
    
        if len(masked_mag):
            ax1.set_ylim(max_mag + .2, min_mag - .2)
            
        ax1.set_xlim(-0.05,2.05)
    
        plt.suptitle(f"DiaObjectId - {diaId}",fontsize=20)
        plt.tight_layout()
    
        if path:
            plt.savefig(f"{path}/{band}_first_lomb.png",dpi=330,bbox_inches="tight")
            pass
        else:
            plt.show()
    
        plt.close()

    return obj_period.value

In [12]:
def second_fit(row,group_data,period0,pick_band="r",path="",filt=100):
    results = group_data.reset_index()

    diaId = row["diaObjectId"]
    
    my_id = row["diaObjectId"]
    
    min_period = period0 - 0.2
    max_period = period0 + 0.2

    if min_period < 0:
        min_period = 0.05

    min_period = min_period * u.day
    max_period = max_period * u.day

    min_freq = 1.0 / max_period
    max_freq = 1.0 / min_period

    # Select the star based on its ID, and measurements in the requested band
    results_band = results[results['band'] == pick_band]

    # Run Lomb-Scargle and pick the "best" frequency
    obj_mjd_days_band = np.array(results_band['expMidptMJD']) * u.day
    obj_fluxes_band = np.array(results_band['psfFlux'])
    obj_frequency, obj_power =\
        LombScargle(obj_mjd_days_band, obj_fluxes_band).autopower(minimum_frequency=min_freq,
                                                                  maximum_frequency=max_freq,
                                                                  samples_per_peak=2000)
    peakbin = np.argmax(obj_power)
    mean_peak_freq = obj_frequency[peakbin].value

    """
    Lomb Results
    """

    # Plot the frequency and power from Lomb-Scargle
    fig, ax = plt.subplots(1, 2, figsize=(7, 4))

    plt.sca(ax[0])
    plt.plot(obj_frequency, obj_power)
    plt.vlines(mean_peak_freq, 0, 0.3, linestyle='--', color='red')
    plt.minorticks_on()
    plt.xlabel('Frequency (1/d)',fontsize=14)
    plt.ylabel('Power',fontsize=14)

    plt.sca(ax[1])
    plt.plot(1 / obj_frequency, obj_power)
    plt.vlines(1/mean_peak_freq, 0, 0.3, linestyle='--', color='red')
    plt.minorticks_on()
    plt.xlabel('Period (d)',fontsize=14)
    plt.ylabel('Power',fontsize=14,rotation=-90,labelpad=15)

    ax[1].yaxis.set_label_position('right')
    ax[1].yaxis.tick_right()

    ax[0].set_ylim(0,np.max(obj_power) + 0.1)

    ax[1].set_ylim(0,np.max(obj_power) + 0.1)

    plt.suptitle(f"DiaObjectId - {diaId}",fontsize=18)
    # plt.tight_layout()
    
    if path:
        plt.savefig(f"{path}/second_lomb_fit_results.png",dpi=330,bbox_inches="tight")
    else:
        plt.show()
    plt.close()

    max_power = np.argmax(obj_power)
    obj_freq = obj_frequency[max_power]
    obj_period = 1.0 / obj_freq

    t0 = 0.0
    obj_mjd_days = np.array(results['expMidptMJD']) * u.day
    obj_fluxes = np.array(results['psfFlux'])

    obj_mjd_norm = (obj_mjd_days.value - t0) / obj_period.value
    obj_phase = np.mod(obj_mjd_norm, 1.0)

    phase = pd.Series(obj_phase)

    mag = -2.5 * np.log10(pd.Series(obj_fluxes) / (3631 * 1e9))

    """
    Saving Combined
    """

    fig, ax = plt.subplots(7, 1, figsize=(7, 21))

    
    max_mag = float("-inf")
    min_mag = float("inf")
    
    ax1 = ax[0]
    for band in filter_names:
        pickband = results['band'] == band
        temp_mag = mag[pickband]

        if len(temp_mag):
            med = np.median(temp_mag)
            std3 = np.std(temp_mag) * filt
        else:
            med = 0
            std3 = 0
    
        mask = (temp_mag > med - std3) & (temp_mag < med + std3)
    
        masked_mag = temp_mag[mask]
    
        FluxErr = results[pickband]["psfFluxErr"][mask]
        FluxBand = results[pickband]["psfFlux"][mask]
    
        yerr = (2.5 / np.log(10)) * (FluxErr / FluxBand)
    
        full_phase = pd.concat([phase[pickband][mask],phase[pickband][mask] + 1])
        full_mag = pd.concat([masked_mag,masked_mag])
        full_mag_err = pd.concat([yerr,yerr])
        
        ax1.errorbar(full_phase, full_mag, yerr=full_mag_err,
                marker=filter_symbols[band], color=filter_colors[band],
                label=f"{band}", markersize=7, linestyle='none', fillstyle='none')
        
        if np.max(masked_mag) > max_mag:
            max_mag = np.max(masked_mag)
    
        if np.min(masked_mag) < min_mag:
            min_mag = np.min(masked_mag) 
        
    
    ax1.set_ylabel('PSF magnitude',fontsize=14)
    ax1.set_xlabel('Phase',fontsize=14)
    ax1.set_title(f"Overall Mag Phase Curve (Filtered)",fontsize=16)
    ax1.minorticks_on()
    ax1.legend(ncols=2, loc='lower left')

    if len(masked_mag):
        ax1.set_ylim(max_mag + .2, min_mag - .2)
        
    ax1.set_xlim(-0.05,2.05)

    
    for i in range(1,7):
        ax1 = ax[i]
    
        band = filter_names[i-1]
    
        pickband = results['band'] == band
        temp_mag = mag[pickband]
    
        if len(temp_mag):
            med = np.median(temp_mag)
            std3 = np.std(temp_mag) * filt
        else:
            med = 0
            std3 = 0
    
        mask = (temp_mag > med - std3) & (temp_mag < med + std3)
    
        masked_mag = temp_mag[mask]
    
        FluxErr = results[pickband]["psfFluxErr"][mask]
        FluxBand = results[pickband]["psfFlux"][mask]
    
        yerr = (2.5 / np.log(10)) * (FluxErr / FluxBand)
    
        full_phase = pd.concat([phase[pickband][mask],phase[pickband][mask] + 1])
        full_mag = pd.concat([masked_mag,masked_mag])
        full_mag_err = pd.concat([yerr,yerr])
        
        ax1.errorbar(full_phase, full_mag, yerr=full_mag_err,
                marker=filter_symbols[band], color=filter_colors[band],
                label=f"{band}", markersize=7, linestyle='none', fillstyle='none')

        if len(masked_mag):
            ax1.set_ylim(np.max(masked_mag) + .2, np.min(masked_mag) - .2)
        
        ax1.set_xlim(-0.05,2.05)
        
        ax1.set_ylabel('PSF magnitude',fontsize=14)
        ax1.set_xlabel('Phase',fontsize=14)
        ax1.set_title(f"{band}-band Mag Phase Curve (Filtered)",fontsize=16)
        ax1.minorticks_on()
        ax1.legend(ncols=2, loc='lower left')
        
    plt.tight_layout()
    plt.suptitle(f"DiaObjectId - {diaId}",fontsize=20,y=1.01)

    if path:
        plt.savefig(f"{path}/combined_second_lomb.png",dpi=330,bbox_inches="tight")
        pass
    else:
        plt.show()

    plt.close()
    
    """
    Saving Overall Version
    """
    
    fig, ax1 = plt.subplots(1, 1, figsize=(7, 4))

    max_mag = float("-inf")
    min_mag = float("inf")
    
    for band in filter_names:
        pickband = results['band'] == band
        temp_mag = mag[pickband]

        if len(temp_mag):
            med = np.median(temp_mag)
            std3 = np.std(temp_mag) * filt
        else:
            med = 0
            std3 = 0
    
        mask = (temp_mag > med - std3) & (temp_mag < med + std3)
    
        masked_mag = temp_mag[mask]

        FluxErr = results[pickband]["psfFluxErr"][mask]
        FluxBand = results[pickband]["psfFlux"][mask]

        yerr = (2.5 / np.log(10)) * (FluxErr / FluxBand)

        full_phase = pd.concat([phase[pickband][mask],phase[pickband][mask] + 1])
        full_mag = pd.concat([masked_mag,masked_mag])
        full_mag_err = pd.concat([yerr,yerr])

        ax1.errorbar(full_phase, full_mag, yerr=full_mag_err,
                marker=filter_symbols[band], color=filter_colors[band],
                label=f"{band}", markersize=7, linestyle='none', fillstyle='none')
        
        if np.max(masked_mag) > max_mag:
            max_mag = np.max(masked_mag)
    
        if np.min(masked_mag) < min_mag:
            min_mag = np.min(masked_mag)

    
    ax1.set_ylabel('PSF magnitude',fontsize=14)
    ax1.set_xlabel('Phase',fontsize=14)
    ax1.set_title(f"Overall Mag Phase Curve (Filtered)",fontsize=16)
    ax1.minorticks_on()
    ax1.legend(ncols=2, loc='lower left')  


    if len(masked_mag):
        ax1.set_ylim(max_mag + .2, min_mag - .2)
        
    ax1.set_xlim(-0.05,2.05)

    plt.suptitle(f"DiaObjectId - {diaId}",fontsize=20)
    plt.tight_layout()

    if path:
        plt.savefig(f"{path}/overall_second_lomb.png",dpi=330,bbox_inches="tight")
        pass
    else:
        plt.show()

    plt.close()

    """
    Saving Individual Bands
    """
    
    for band in filter_names:
        fig, ax1 = plt.subplots(1, 1, figsize=(7, 4))

        max_mag = float("-inf")
        min_mag = float("inf")
        
        pickband = results['band'] == band
        temp_mag = mag[pickband]

        if len(temp_mag):
            med = np.median(temp_mag)
            std3 = np.std(temp_mag) * filt
        else:
            med = 0
            std3 = 0
    
        mask = (temp_mag > med - std3) & (temp_mag < med + std3)
    
        masked_mag = temp_mag[mask]

        FluxErr = results[pickband]["psfFluxErr"][mask]
        FluxBand = results[pickband]["psfFlux"][mask]

        yerr = (2.5 / np.log(10)) * (FluxErr / FluxBand)

        full_phase = pd.concat([phase[pickband][mask],phase[pickband][mask] + 1])
        full_mag = pd.concat([masked_mag,masked_mag])
        full_mag_err = pd.concat([yerr,yerr])

        ax1.errorbar(full_phase, full_mag, yerr=full_mag_err,
                marker=filter_symbols[band], color=filter_colors[band],
                label=f"{band}", markersize=7, linestyle='none', fillstyle='none')
        
        if np.max(masked_mag) > max_mag:
            max_mag = np.max(masked_mag)
    
        if np.min(masked_mag) < min_mag:
            min_mag = np.min(masked_mag)

    
        ax1.set_ylabel('PSF magnitude',fontsize=14)
        ax1.set_xlabel('Phase',fontsize=14)
        ax1.set_title(f"Overall Mag Phase Curve (Filtered)",fontsize=16)
        ax1.minorticks_on()
        ax1.legend(ncols=2, loc='lower left')  
    
    
        if len(masked_mag):
            ax1.set_ylim(max_mag + .2, min_mag - .2)
            
        ax1.set_xlim(-0.05,2.05)
    
        plt.suptitle(f"DiaObjectId - {diaId}",fontsize=20)
        plt.tight_layout()
    
        if path:
            plt.savefig(f"{path}/{band}_second_lomb.png",dpi=330,bbox_inches="tight")
            pass
        else:
            plt.show()
    
        plt.close()

    return obj_period.value

In [13]:
def make_plots(row,path=""):
    print(f"Creating LombScargle plots for {row["diaObjectId"]}...")

    #pull out forcedDiaSources associated with current star
    # group_data = grouped_forcedSearch.get_group(row["ObjectId"])
    group_data = grouped_diaSearch.get_group(row["diaObjectId"])

    
    #filted out any negative flux values
    group_data = group_data[group_data['psfFlux'] > 0]

    #mask to remove all bad pixels or saturated measurements from sources
    back_sat = ~group_data["pixelFlags_saturated"]
    cent_sat = ~group_data["pixelFlags_saturatedCenter"]
    bad_pixel = ~group_data["pixelFlags_bad"]

    #update group data based on mask
    group_data = group_data[back_sat & cent_sat & bad_pixel]

    #save star diaObjectId
    diaId = row["diaObjectId"]

    if path:
        # Creating New Folder for current star
        dia_path = os.path.join(path,f"{diaId}")
        
        if not os.path.isdir(dia_path):
            os.mkdir(dia_path)

    if path:
        # Creating New Folder for days plots
        lomb_path = os.path.join(dia_path,"lomb")
        
        if not os.path.isdir(lomb_path):
            os.mkdir(lomb_path)

    p0 =  first_fit(row,group_data,path=lomb_path)

    band_dict = {}
    for band in filter_names:
        band_dict[band] = np.sum((row[f"nGood{band}"]))
    
    max_band = sorted(band_dict.items(),key=lambda x:x[-1])[-1][0]

    p1 = second_fit(row,group_data,p0,pick_band=max_band,path=lomb_path)

    return p0, p1

## Defining Days Plots

In [14]:
def days_flux_plots(star_group, diaId, directory=""):    
    fig, ax = plt.subplots(7, 2, figsize=(14, 21))
    fig.subplots_adjust(wspace=0)
    
    ax1 = ax[0,0]
    ax2 = ax[0,1]
    for band in filter_names:
        pickband = star_group['band'] == band
        ax1.errorbar(star_group[pickband]['expMidptMJD'], star_group[pickband]['psfDiffFlux'],
                     yerr=star_group[pickband]["psfDiffFluxErr"],
                     marker=filter_symbols[band], color=filter_colors[band],
                     label=f"{band}", markersize=7, linestyle='none', fillstyle='none')
        ax2.errorbar(star_group[pickband]['expMidptMJD'], star_group[pickband]['psfFlux'],
                 yerr=star_group[pickband]["psfFluxErr"],
                 marker=filter_symbols[band], color=filter_colors[band], linestyle='none',
                 markersize=7, fillstyle='none')
    
    ax2.yaxis.tick_right()
    ax2.yaxis.set_label_position('right')

    ax1.tick_params(labelsize=10)
    ax2.tick_params(labelsize=10)
    
    ax1.set_ylabel('PSF difference\nimage flux (nJy)',fontsize=14)
    ax2.set_ylabel('PSF direct\nimage flux (nJy)',rotation=-90,labelpad=35,fontsize=14)
    ax1.set_xlabel('Exposure midpoint, MJD (d)',fontsize=14)
    ax2.set_xlabel('Exposure midpoint, MJD (d)',fontsize=14)

    ax1.set_title(f"Overall DiffFlux MJD Curve",fontsize=16)
    ax2.set_title(f"Overall Flux MJD Curve",fontsize=16)

    ax1.set_xlim(np.min(star_group['expMidptMJD']) - .5,np.max(star_group['expMidptMJD'] + .5))
    ax2.set_xlim(np.min(star_group['expMidptMJD']) - .5,np.max(star_group['expMidptMJD'] + .5))
    
    ax1.minorticks_on()
    ax2.minorticks_on()
    ax1.legend(ncols=2,loc="upper left")
    
    for i in range(1,7):
        ax1 = ax[i,0]
        ax2 = ax[i,1]
    
        band = filter_names[i-1]
    
        pickband = star_group['band'] == band
        ax1.errorbar(star_group[pickband]['expMidptMJD'], star_group[pickband]['psfDiffFlux'],
                     yerr=star_group[pickband]["psfDiffFluxErr"],
                     marker=filter_symbols[band], color=filter_colors[band],
                     label=f"{band}", markersize=7, linestyle='none', fillstyle='none')
        ax2.errorbar(star_group[pickband]['expMidptMJD'], star_group[pickband]['psfFlux'],
                 yerr=star_group[pickband]["psfFluxErr"],
                 marker=filter_symbols[band], color=filter_colors[band], linestyle='none',
                 markersize=7, fillstyle='none')
    
        ax2.yaxis.tick_right()
        ax2.yaxis.set_label_position('right')
        
        ax1.set_ylabel('PSF difference\nimage flux (nJy)',fontsize=14)
        ax2.set_ylabel('PSF direct\nimage flux (nJy)',rotation=-90,labelpad=35,fontsize=14)
        ax1.set_xlabel('Exposure midpoint, MJD (d)',fontsize=14)
        ax2.set_xlabel('Exposure midpoint, MJD (d)',fontsize=14)

        ax1.set_title(f"{band}-band DiffFlux MJD Curve",fontsize=16)
        ax2.set_title(f"{band}-band Flux MJD Curve",fontsize=16)

        ax1.tick_params(labelsize=10)
        ax2.tick_params(labelsize=10)

        ax1.set_xlim(np.min(star_group['expMidptMJD']) - .5,np.max(star_group['expMidptMJD'] + .5))
        ax2.set_xlim(np.min(star_group['expMidptMJD']) - .5,np.max(star_group['expMidptMJD'] + .5))
        
        ax1.minorticks_on()
        ax2.minorticks_on()
        ax1.legend(ncols=2,loc="upper left")

    plt.ticklabel_format(style='plain', axis='y')
    plt.tight_layout()
    plt.suptitle(f"DiaObjectId - {diaId}",fontsize=20,y=1.01) #putting after tight layout makes each more uniform
    
    if directory:
        plt.savefig(f"{directory}/combined_days.png",dpi=330,bbox_inches="tight")
        pass
    else:
        plt.show()
    plt.close()

"""
================================================================================================================
Overall Above ^

Individual Below v
================================================================================================================
"""

def individ_days_flux_plots(star_group,diaId,directory=""):
    fig, ax = plt.subplots(1, 2, figsize=(14, 4), sharex=True)
    fig.subplots_adjust(wspace=0)
    
    ax1 = ax[0]
    ax2 = ax[1]
    for band in filter_names:
        pickband = star_group['band'] == band
        ax1.errorbar(star_group[pickband]['expMidptMJD'], star_group[pickband]['psfDiffFlux'],
                     yerr=star_group[pickband]["psfDiffFluxErr"],
                     marker=filter_symbols[band], color=filter_colors[band],
                     label=f"{band}", markersize=7, linestyle='none', fillstyle='none')
        ax2.errorbar(star_group[pickband]['expMidptMJD'], star_group[pickband]['psfFlux'],
                 yerr=star_group[pickband]["psfFluxErr"],
                 marker=filter_symbols[band], color=filter_colors[band], linestyle='none',
                 markersize=7, fillstyle='none')
    
    ax2.yaxis.tick_right()
    ax2.yaxis.set_label_position('right')

    ax1.tick_params(labelsize=10)
    ax2.tick_params(labelsize=10)
    
    ax1.set_ylabel('PSF difference\nimage flux (nJy)',fontsize=14)
    ax2.set_ylabel('PSF direct\nimage flux (nJy)',rotation=-90,labelpad=35,fontsize=14)
    ax1.set_xlabel('Exposure midpoint, MJD (d)',fontsize=14)
    ax2.set_xlabel('Exposure midpoint, MJD (d)',fontsize=14)

    ax1.set_title(f"Overall DiffFlux MJD Curve",fontsize=16)
    ax2.set_title(f"Overall Flux MJD Curve",fontsize=16)

    ax1.set_xlim(np.min(star_group['expMidptMJD']) - .5,np.max(star_group['expMidptMJD'] + .5))
    ax2.set_xlim(np.min(star_group['expMidptMJD']) - .5,np.max(star_group['expMidptMJD'] + .5))
    
    ax1.minorticks_on()
    ax2.minorticks_on()
    ax1.legend(ncols=2,loc="upper left")
    
    plt.suptitle(f"DiaObjectId - {diaId}",fontsize=20)
    plt.tight_layout()
    
    if directory:
        plt.savefig(f"{directory}/overall_days.png",dpi=330,bbox_inches="tight")
        pass
    else:
        plt.show()
    plt.close()
    
    for i in range(1,7):
        fig, ax = plt.subplots(1, 2, figsize=(14, 4), sharex=True)
        fig.subplots_adjust(wspace=0)
        
        ax1 = ax[0]
        ax2 = ax[1]
    
        band = filter_names[i-1]
    
        pickband = star_group['band'] == band
        ax1.errorbar(star_group[pickband]['expMidptMJD'], star_group[pickband]['psfDiffFlux'],
                     yerr=star_group[pickband]["psfDiffFluxErr"],
                     marker=filter_symbols[band], color=filter_colors[band],
                     label=f"{band}", markersize=7, linestyle='none', fillstyle='none')
        ax2.errorbar(star_group[pickband]['expMidptMJD'], star_group[pickband]['psfFlux'],
                 yerr=star_group[pickband]["psfFluxErr"],
                 marker=filter_symbols[band], color=filter_colors[band], linestyle='none',
                 markersize=7, fillstyle='none')
    
        ax2.yaxis.tick_right()
        ax2.yaxis.set_label_position('right')
        
        ax1.set_ylabel('PSF difference\nimage flux (nJy)',fontsize=14)
        ax2.set_ylabel('PSF direct\nimage flux (nJy)',rotation=-90,labelpad=35,fontsize=14)
        ax1.set_xlabel('Exposure midpoint, MJD (d)',fontsize=14)
        ax2.set_xlabel('Exposure midpoint, MJD (d)',fontsize=14)

        ax1.set_title(f"{band}-band DiffFlux MJD Curve",fontsize=16)
        ax2.set_title(f"{band}-band Flux MJD Curve",fontsize=16)

        ax1.tick_params(labelsize=10)
        ax2.tick_params(labelsize=10)

        ax1.set_xlim(np.min(star_group['expMidptMJD']) - .5,np.max(star_group['expMidptMJD'] + .5))
        ax2.set_xlim(np.min(star_group['expMidptMJD']) - .5,np.max(star_group['expMidptMJD'] + .5))
        
        ax1.minorticks_on()
        ax2.minorticks_on()
        ax1.legend(ncols=2,loc="upper left")
        plt.suptitle(f"DiaObjectId - {diaId}",fontsize=20)
        plt.ticklabel_format(style='plain', axis='y')
        plt.tight_layout()
    
        if directory:
            plt.savefig(f"{directory}/{band}_days.png",dpi=330,bbox_inches="tight")            
            pass
        else:
            plt.show()
        plt.close()

In [15]:
def create_plots(row,path="",filt=100):
    print(f"Making MJD plots for {row["diaObjectId"]}...")

    #pull out forcedDiaSources associated with current star
    group_data = grouped_diaSearch.get_group(row["diaObjectId"])
    #filted out any negative flux values
    group_data = group_data[group_data['psfFlux'] > 0]

    #mask to remove all bad pixels or saturated measurements from sources
    back_sat = ~group_data["pixelFlags_saturated"]
    cent_sat = ~group_data["pixelFlags_saturatedCenter"]
    bad_pixel = ~group_data["pixelFlags_bad"]

    #update group data based on mask
    group_data = group_data[back_sat & cent_sat & bad_pixel]

    #save star diaObjectId
    diaId = row["diaObjectId"]

    if path:
        # Creating New Folder for current star
        dia_path = os.path.join(path,f"{diaId}")
        
        if not os.path.isdir(dia_path):
            os.mkdir(dia_path)

    """
    Plot/Save Flux Curves over MJD
    """

    if path:
        # Creating New Folder for days plots
        days_path = os.path.join(dia_path,"days")
        
        if not os.path.isdir(days_path):
            os.mkdir(days_path)
            
    days_flux_plots(group_data,diaId,directory=f"{days_path}")
    
    individ_days_flux_plots(group_data,diaId,directory=f"{days_path}")

## Defining Table Functions

In [16]:
def save_row_as_fig(row,lomb_df="",path=""):

    try:
        lomb_p_num = lomb_df[lomb_df["diaObjectId"] == row['diaObjectId']]["lomb_p1"].values[0]
        lomb_p = f"{lomb_p_num:.4f}"
    except:
        lomb_p = "N/A"


    info_table = [
        ['diaObjectId', 'ObjectId', "Simbad_Class"],
        [row['diaObjectId'],row['ObjectId'],row["simbad_results"]],
        ['Field','RA, Dec (deg)',"Classification"],
        [row['DR3_field_loc_center'],f"{row['ra']:.4f}, {row['dec']:.4f}","N/A"],
        ['Mean r mag (dia)',"Mean r mag (Obj)",'Total DiaSources'],
        [f"{row['filt_mean_r_mag']:.4f}",f"{row['r_psfMag']:.4f}",row['nDiaSources']],
        ['r mag range (d)',"LombScargle Period","Total Good Sources"],
        [f"{row['mag_var']:.4f}",lomb_p,row["nGoodSources"]],
        ['nGoodu','nGoodg','nGoodr'],
        [row['nGoodu'],row['nGoodg'],row['nGoodr']],
        ['nGoodi','nGoody','nGoodz'],
        [row['nGoodi'],row['nGoody'],row['nGoodz']]
    ]
        
        
        
    
    # Create figure and axis
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('tight')
    ax.axis('off')
    
    # Create the table
    table = ax.table(cellText=info_table, loc='center', cellLoc='left')
    
    # Styling: Set font size and scale the table
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.8)

    
    for j in range(0,len(info_table),2):
        for i in range(3):
            cell = table[j,i]
            cell.set_facecolor('rebeccapurple')
            cell.set_text_props(color='white', weight='bold')
            cell.get_text().set_weight('bold')
    
    for j in range(1,len(info_table),2):
        for i in range(3):
            cell = table[j,i]
            cell.set_facecolor('w')
            cell.set_text_props(color='k')

    plt.title(f"Stellar Information on {row['diaObjectId']}", fontsize=20, weight='bold',y=.9)
    plt.tight_layout()
    if path:
        plt.savefig(f"{path}/info_table.png",dpi=330,bbox_inches="tight")
    else:
        plt.show()
    plt.close()
    pass

In [17]:
def filter_table(row,lomb_df="",path=""):
    print(f"Creating Table plot for {row["diaObjectId"]}...")
    
    wanted_columns =["diaObjectId","ObjectId","ra","dec",
                     "filt_mean_r_mag","DR3_field_loc_center",
                     "nDiaSources","nGoodSources",
                     "nGoodu","nGoodg","nGoodr","nGoodi","nGoodz","nGoody",
                     "query","simbad_results","r_psfMag","mag_var"]

    #may want to use DR3_phot_g_mean_flux and DR3_phot_g_mean_flux_error instead
    # if row["DR3_best_classification"] == "RRab" or row["DR3_best_classification"] == "RRd":
    #     wanted_columns.append("DR3_pf")
    #     wanted_columns.append("DR3_pf_error")
    # if row["DR3_best_classification"] == "RRc" or row["DR3_best_classification"] == "RRd":
    #     wanted_columns.append("DR3_p1_o")
    #     wanted_columns.append("DR3_p1_o_error")
        

    if path:
        # Creating New Folder for current star
        info_path = os.path.join(path,f"{row["diaObjectId"]}")
        
        if not os.path.isdir(info_path):
            os.mkdir(info_path)
    else:
        info_path = ""


    save_row_as_fig(row[wanted_columns],lomb_df,path=info_path)
        
                     
    pass

## Main Function

In [18]:
def main():

    outdir = os.getcwd()
    auto_path = os.path.join(outdir,"auto_rrl_search")
    
    if not os.path.isdir(auto_path):
        os.mkdir(auto_path)

    search_df.apply(create_plots, path=auto_path, axis=1)

    #Creating loc plots for star
    search_df.apply(pull_out_row, path=auto_path, axis=1)


    #Creating lomb fits for star
    ps = search_df.apply(make_plots,path=auto_path,axis=1)

    c1 = []
    c2 = []
    for row in ps.values:
        c1.append(row[0])
        c2.append(row[1])
    
    df = pd.DataFrame({"diaObjectId":search_df["diaObjectId"],"lomb_p0":c1,"lomb_p1":c2})

    df.to_csv("data/cc_lomb_diaObj_ps.csv")

    df = pd.read_csv("data/cc_lomb_diaObj_ps.csv")

    #Creating Info Table for star
    search_df.apply(filter_table,lomb_df=df,path=auto_path,axis=1)
    
    pass

In [19]:
main()

Making MJD plots for 614435753623027782...
Making MJD plots for 614436372098318338...
Making MJD plots for 614436372098318441...
Making MJD plots for 614437677768376364...
Making MJD plots for 614430393503842317...
Making MJD plots for 614436234659364872...
Making MJD plots for 614436990573608968...
Making MJD plots for 612929972448788650...
Making MJD plots for 614436921854132236...
Making MJD plots for 614437677768376338...
Making MJD plots for 614436921854132316...
Making MJD plots for 614436921854132251...
Making MJD plots for 614436853134655524...
Making MJD plots for 614437540329422849...
Making MJD plots for 614439052157911316...
Making location plots for 614435753623027782...
Making location plots for 614436372098318338...
Making location plots for 614436372098318441...
Making location plots for 614437677768376364...
Making location plots for 614430393503842317...
Making location plots for 614436234659364872...
Making location plots for 614436990573608968...
Making location plo